# Replaying experimental recordings and inferring unobserved quantities

TODO: intro

## Load and preprocess experimental recordings

TODO: explain

In [ ]:
import numpy as np
from flygym import assets_dir

demo_data_path = assets_dir / "demo/aymans2022_behavior_clip.npz"
demo_data = np.load(demo_data_path)

print("Data dimensions:", demo_data["dof_angles"].shape)
print("DoFs:", *demo_data["dof_order"][:3], "...")
print("FPS:", demo_data["fps"])
print("Source file:", demo_data["source_file"])
print("Frame range:", demo_data["frame_range"])
print("Unit:", demo_data["unit"])
print("Axis order:", demo_data["axis_order"])

The experimental data was recorded at 100 Hz, but we want to simulate at a timestep of 0.0001 s for accurate physics. This means that we need to interpolate the experimental data:

In [ ]:
sim_timestep = 1e-4

n_steps_in = demo_data["dof_angles"].shape[0]
interp_factor = int(1 / (demo_data["fps"] * sim_timestep))
n_steps_out = (n_steps_in - 1) * interp_factor

print(f"Input frames: {n_steps_in}; output frames after interpolation: {n_steps_out}")

In [ ]:
from scipy.interpolate import interp1d

times_before_interp = np.arange(n_steps_in) / demo_data["fps"]
times = np.arange(n_steps_out) * sim_timestep
interpolator = interp1d(times_before_interp, demo_data["dof_angles"], axis=0)
dof_angles = interpolator(times)

print("Interpolated data shape:", dof_angles.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3), tight_layout=True)
for dof_id in range(7):
    angles_time_series = np.rad2deg(dof_angles[:, dof_id])
    dof_name = demo_data["dof_order"][dof_id]
    ax.plot(times, angles_time_series, label=dof_name)
ax.legend(bbox_to_anchor=(1.04, 1), borderaxespad=0)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Angle (deg)")
ax.set_title("Joint angles over time (one leg only)")

## Composing Fly and World model

This part essentially repeats the content of the tutorial "Basic Model Composition."

In [ ]:
from flygym.compose import Fly, ActuatorType, FlatGroundWorld
from flygym.anatomy import Skeleton, JointPreset, ActuatedDOFPreset, AxisOrder
from flygym.compose.pose import KinematicPose
from flygym.utils.math import Rotation3D

joints_preset = JointPreset.LEGS_ONLY
actuated_dofs_preset = ActuatedDOFPreset.LEGS_ACTIVE_ONLY
actuator_type = ActuatorType.POSITION
position_gain = 50.0
neutral_pose_file = assets_dir / "model/pose/neutral.yaml"
spawn_position = (0, 0, 0.7)  # xyz in mm
spawn_rotation = Rotation3D("quat", (1, 0, 0, 0))

fly = Fly()
axis_order = AxisOrder.YAW_PITCH_ROLL

# Add joints
skeleton = Skeleton(axis_order=axis_order, joint_preset=joints_preset)
neutral_pose = KinematicPose(path=neutral_pose_file)
fly.add_joints(skeleton, neutral_pose=neutral_pose)

# Add position actuators and set them to the neutral pose
actuated_dofs_list = fly.skeleton.get_actuated_dofs_from_preset(actuated_dofs_preset)
fly.add_actuators(
    actuated_dofs_list,
    actuator_type=actuator_type,
    kp=position_gain,
    neutral_input=neutral_pose,
)

# Add visuals
fly.colorize()
cam = fly.add_tracking_camera()

# Create a world and spawn the fly
world = FlatGroundWorld()
world.add_fly(fly, spawn_position, spawn_rotation)

## Creating a Simulation

In [ ]:
from flygym import Simulation

sim = Simulation(world)

Add a renderer (more information below):

In [ ]:
playback_speed = 0.2  # rendered video will be played at 20% speed
output_fps = 25  # ... and when played at that speed, the video will have 25 FPS

# The render will automatically figure out that this means it should render
# 25 / 0.2 = 125 frames per simulated second internally.
renderer_name = sim.add_renderer(
    name="trackingcam", camera=cam, playback_speed=playback_speed, output_fps=output_fps
)

The orderings of DoFs and actuators internally within MuJoCo are generally not the same as the orders by which they are specified in the MJCF model. Therefore, it is essential that we access states and set actuator inputs in the correct order.

Reordering the elements is an annoying process—which is why we have encapsulated it into the `Simulation` class. As user, you should get the FlyGym canonical DoF/actuator ordering using `Fly.get_jointdofs_order()` and `Fly.get_actuated_jointdofs_order(actuator_type)`, and always assume that simulation data are ordered as such.

In [ ]:
dof_order_in_experimental_data = list(demo_data["dof_order"])

print("DoF order in experimental recording:")
print(" ", ",".join(dof_order_in_experimental_data[:3]), "...")

dof_order_in_sim = [dof.name for dof in fly.get_jointdofs_order()]
actuator_order_in_sim = [
    dof.name for dof in fly.get_actuated_jointdofs_order(actuator_type)
]

print("\nFlyGym canonical DoF order (by fly):")
print(" ", ",".join(dof_order_in_sim[:3]), "...")
print("FlyGym actuated DoF order (by fly and actuator type):")
print(" ", ",".join(actuator_order_in_sim[:3]), "...")

Reorder columns in experimental data so that the DoF order matches the actuator order in simulation:

In [ ]:
ids_data2simactuator = [
    dof_order_in_experimental_data.index(dof) for dof in actuator_order_in_sim
]

print("Indices to map DoFs in experimental recording to sim actuators:")
print(" ", ", ".join(map(str, ids_data2simactuator[:10])), "...")

dof_angles_reordered = dof_angles[:, ids_data2simactuator]

Next, define how long we want the simulation to be. We will add a warm-up period during which the fly stays at the neutral pose; replay of experimental data will only start after the fly has "landed" on the ground and the transient forces have been settled:

In [ ]:
warmup_period = 0.05  # seconds
time_to_simulate = 5  # seconds

warmup_steps = int(warmup_period / sim_timestep)
sim_steps = int(time_to_simulate / sim_timestep)

assert sim_steps <= dof_angles_reordered.shape[0], "Not enough data to simulate"

Let's implement the simulation loop. See inline comments for explanations.

In [ ]:
from tqdm import trange  # for progress bar

for step in trange(warmup_steps + sim_steps):
    # Set control input only after the warmup period
    if step >= warmup_steps:
        # Find corresponding timestep in (interpolated) experimental data
        control_inputs = dof_angles_reordered[step - warmup_steps, :]
        # Set actuator control inputs in the sim (in this case just target joint angles)
        sim.set_actuator_inputs(fly.name, actuator_type, control_inputs)
    
    # Step the simulation forward
    sim.step()
    
    # The FlyGym renderer automatically decides whether to render at each step based on
    # the playback speed and output video FPS requested by the user. Just call
    # `render_as_needed()` at every step and let it handle this internally.
    sim.render_as_needed()

In [ ]:
sim.renderers[renderer_name].show_in_notebook()

A handy `.print_performance_report()` method is built into FlyGym to monitor performance stats:

In [ ]:
sim.print_performance_report()

## Reading out unserved quantities from simulation

TODO